# DSN Mart Sales Prediction

Predict `total_sales` for product-store combinations using preprocessing, feature engineering, and a Random Forest regression model.

## 1. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

RANDOM_STATE = 42

## 2. Load and combine train/test data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['is_train'] = 1
test['is_train'] = 0
test['total_sales'] = np.nan

full = pd.concat([train, test], sort=False, ignore_index=True)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Combined shape:", full.shape)

## 3. Clean inconsistent text categories

In [ ]:
full['product_category'] = full['product_category'].str.strip().str.title()
full['fat_content'] = full['fat_content'].str.strip().str.title()

## 4. Impute missing values

In [ ]:
# Product weight: use the median weight of the same product elsewhere
prod_weight = full.groupby('product_code')['product_weight_kg'].transform('median')
full['product_weight_kg'] = full['product_weight_kg'].fillna(prod_weight)

# Fallback: use the median weight for the product category
cat_weight = full.groupby('product_category')['product_weight_kg'].transform('median')
full['product_weight_kg'] = full['product_weight_kg'].fillna(cat_weight)

# Store size: missing for entire stores, so retain "Unknown" as its own category
full['store_size'] = full['store_size'].fillna('Unknown')

# Flag exact-zero shelf visibility
full['visibility_was_zero'] = (full['shelf_visibility'] == 0).astype(int)

print("Missing values after imputation:")
print(full.isna().sum())

## 5. Feature engineering

In [ ]:
full['price_per_kg'] = full['product_price'] / full['product_weight_kg']
full['store_profile'] = full['store_format'] + '_' + full['store_size']

## 6. Define features and preprocessing

In [ ]:
numeric_cols = [
    'product_weight_kg',
    'shelf_visibility',
    'product_price',
    'store_age_years',
    'visibility_was_zero',
    'price_per_kg'
]

ordinal_cols = ['store_size', 'store_location_tier']

nominal_cols = [
    'fat_content',
    'product_category',
    'store_code',
    'store_format',
    'store_profile'
]

features = numeric_cols + ordinal_cols + nominal_cols

size_order = ['Unknown', 'Small', 'Medium', 'Large']
tier_order = ['Tier_1', 'Tier_2', 'Tier_3']

preprocess = ColumnTransformer([
    ('num', 'passthrough', numeric_cols),
    ('ord', OrdinalEncoder(categories=[size_order, tier_order]), ordinal_cols),
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
])

## 7. Random Forest model

In [ ]:
model = RandomForestRegressor(
    n_estimators=600,
    max_depth=6,
    min_samples_leaf=8,
    max_features=0.6,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipe = Pipeline([
    ('prep', preprocess),
    ('model', model)
])

## 8. Cross-validation

In [ ]:
train_data = full[full['is_train'] == 1]
X = train_data[features]
y = train_data['total_sales']

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rmse = -cross_val_score(
    pipe,
    X,
    y,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print(f"Cross-validated RMSE: {cv_rmse.mean():.2f} (+/- {cv_rmse.std():.2f})")

## 9. Fit on full training data and predict test data

In [ ]:
pipe.fit(X, y)

test_data = full[full['is_train'] == 0]
preds = pipe.predict(test_data[features])

# Sales cannot be negative
preds = np.clip(preds, 0, None)

submission = pd.DataFrame({
    'id': test_data['id'],
    'total_sales': preds
})

submission.to_csv('/mnt/user-data/outputs/submission.csv', index=False)

print("Saved submission.csv:")
print(submission.head())

## 10. Feature importance

In [ ]:
ohe = pipe.named_steps['prep'].named_transformers_['nom']
ohe_names = ohe.get_feature_names_out(nominal_cols)

all_names = numeric_cols + ordinal_cols + list(ohe_names)

importances = pipe.named_steps['model'].feature_importances_

imp_df = (
    pd.DataFrame({
        'feature': all_names,
        'importance': importances
    })
    .sort_values('importance', ascending=False)
)

print("Top 10 most important features:")
print(imp_df.head(10).to_string(index=False))